# K-Means và PCA (không dùng nhãn khi fit, chỉ fit trên train)
Chỉ dùng cột số liên tục/đếm (bỏ cờ 0/1 và `clicks_pre_start`).

**Số liệu train:** 19.264 dòng (lượt đăng ký) thuộc 17.394 sinh viên duy nhất; một sinh viên có thể đăng ký nhiều module nhưng chỉ thuộc một split. Dev: 4.131 dòng.

In [1]:
import sys, warnings
from pathlib import Path
import pandas as pd
warnings.filterwarnings('ignore')
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.models.train_kmeans import run as run_kmeans
from src.models.fit_pca import run as run_pca

## K-Means
K đã chốt = 3 (elbow gập rõ nhất ở K=3, silhouette còn khá).

In [2]:
scores, best_k, summary, fig = run_kmeans(ROOT, k=3)
scores

,k,inertia,silhouette
0,2,288072.526938,0.378046
1,3,236708.056605,0.301681
2,4,203271.435072,0.223752
3,5,184001.295946,0.221488
4,6,173364.493513,0.191979
5,7,164512.529528,0.192476
6,8,157232.128839,0.193789


In [3]:
from IPython.display import Image
Image(ROOT / 'results/kmeans_elbow_silhouette.png')

**Nhận xét:** silhouette cao nhất ở K=2 (0,38) nhưng giảm nhanh; inertia gập ở K=3 nên chọn K=3.

In [4]:
summary

,cluster,train_n_students,train_withdrawn_rate,train_pass_rate,train_fail_rate,train_distinction_rate,mean_total_clicks,mean_active_days,mean_n_assess_submitted,mean_avg_score,dev_n_students,dev_withdrawn_rate,dev_pass_rate,dev_fail_rate,dev_distinction_rate
0,0,4144,0.196187,0.418919,0.269788,0.115106,198.717181,8.492278,0.023407,0.091216,883,0.187995,0.408834,0.272933,0.130238
1,1,1909,0.238869,0.218963,0.504453,0.037716,5.340492,0.774227,0.180199,11.371399,384,0.218750,0.260417,0.500000,0.020833
2,2,13211,0.175081,0.493831,0.215729,0.115358,339.179472,12.728030,1.047839,73.359587,2864,0.174930,0.479749,0.222416,0.122905


In [5]:
summary.T

,0,1,2
cluster,0.000000,1.000000,2.000000
train_n_students,4144.000000,1909.000000,13211.000000
train_withdrawn_rate,0.196187,0.238869,0.175081
train_pass_rate,0.418919,0.218963,0.493831
train_fail_rate,0.269788,0.504453,0.215729
train_distinction_rate,0.115106,0.037716,0.115358
mean_total_clicks,198.717181,5.340492,339.179472
mean_active_days,8.492278,0.774227,12.728030
mean_n_assess_submitted,0.023407,0.180199,1.047839
mean_avg_score,0.091216,11.371399,73.359587


**Mô tả cụm (K=3, thống kê trên train, dev để kiểm tra):**

- **Cụm 0 – "vào học nhưng không nộp bài" (4.144 dòng):** click trung bình khoảng 199 và học khoảng 8,5 ngày, nhưng gần như chưa nộp bài (0,02 bài). Pass 42%, Fail 27%, Withdrawn 20%.
- **Cụm 1 – "gần như không hoạt động" (1.909 dòng):** chỉ khoảng 5 click và chưa đến 1 ngày học. Fail 50% và Withdrawn 24%, cao nhất; Pass chỉ 22%. Đây là nhóm rủi ro cao nhất.
- **Cụm 2 – "hoạt động đều" (13.211 dòng):** khoảng 339 click, 12,7 ngày học, nộp trung bình 1 bài, điểm trung bình 73. Pass 49%, Withdrawn 17,5% (thấp nhất).

Tỉ lệ nhãn trên dev gần với train. Lưu ý: tỉ lệ Withdrawn giữa các cụm chênh không nhiều (17,5% đến 24%), nên cụm chỉ phân biệt mức hoạt động chứ chưa tách hẳn người rút lui. `final_result` chỉ dùng để thống kê, không làm feature.

**Nhận xét:** cụm 1 (nhỏ nhất) có tỉ lệ Fail 50% và Withdrawn cao nhất (khoảng 24%). Tỉ lệ nhãn trên dev gần với train (ví dụ cụm 1: 22% so với 24%), nên cụm khá ổn định. `final_result` chỉ dùng để thống kê này, không làm feature.

## PCA
Chốt 11 thành phần (≥95% phương sai tích lũy); đã lưu `models/pca.joblib` (kèm danh sách cột theo đúng thứ tự).

In [6]:
table, n_comp, tops, scree = run_pca(ROOT, n_components=11, save=False)
table

,component,eigenvalue,explained,cumulative
0,PC1,10.822237,0.491894,0.491894
1,PC2,2.790173,0.126819,0.618714
2,PC3,1.640024,0.074543,0.693256
3,PC4,1.168228,0.053099,0.746355
4,PC5,0.981496,0.044611,0.790966
5,PC6,0.769060,0.034955,0.825922
6,PC7,0.746196,0.033916,0.859838
7,PC8,0.634578,0.028843,0.888681
8,PC9,0.553043,0.025137,0.913818
9,PC10,0.473504,0.021522,0.935340


In [7]:
Image(ROOT / 'results/pca_scree.png')

**Nhận xét:** PC1 (49%) là mức độ hoạt động chung, PC2 (13%) là bài tập/điểm. 11 thành phần giữ 95,4% phương sai; eigenvalue > 1 chỉ cho 4 thành phần.

In [8]:
pd.DataFrame({pc: [f'{k.split("__")[1]} ({v:+.2f})' for k, v in t.items()] for pc, t in tops.items()})

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11
0,total_clicks (+0.29),n_assess_due (+0.46),click_trend (+0.74),num_of_prev_attempts (+0.70),date_registration (+0.95),studied_credits (+0.64),num_of_prev_attempts (+0.42),clicks_w4 (+0.50),clicks_w3 (+0.48),clicks_w1 (+0.57),n_assess_due (+0.67)
1,active_days (+0.28),n_assess_submitted (+0.42),clicks_w1 (-0.33),studied_credits (+0.58),num_of_prev_attempts (+0.28),num_of_prev_attempts (-0.47),studied_credits (-0.41),clicks_per_active_day (-0.50),weighted_score (-0.45),clicks_w2 (-0.49),weighted_score (-0.39)
2,n_distinct_sites (+0.28),assess_submit_rate (+0.41),clicks_w2 (-0.25),date_registration (-0.27),studied_credits (+0.11),last_active_day (-0.25),weighted_score (+0.41),clicks_w2 (+0.29),clicks_w2 (+0.39),clicks_per_active_day (-0.33),clicks_w4 (+0.36)
3,longest_gap_days (-0.28),avg_score (+0.38),clicks_w3 (+0.24),avg_score (-0.17),click_trend (-0.06),days_since_last_active (+0.25),last_active_day (-0.30),n_distinct_activity_types (-0.27),longest_gap_days (-0.32),first_active_day (-0.31),avg_score (-0.30)
4,n_distinct_activity_types (+0.28),weighted_score (+0.27),first_active_day (+0.23),assess_submit_rate (-0.15),clicks_per_active_day (+0.06),clicks_w3 (+0.25),days_since_last_active (+0.30),clicks_w3 (-0.23),clicks_w4 (-0.31),clicks_w3 (+0.27),clicks_w1 (+0.23)


**Nhận xét:** PC3 chủ yếu là `click_trend`; PC4–PC7 gắn với đặc điểm đăng ký (số lần học trước, tín chỉ, ngày đăng ký).